# Machine + Hydra Data Validation and LSTM Forecast

This notebook does 5 things:

1. Load the two datasets:
   - Machine/Hydra MES data (`MES_Manufacturing_...Data.csv`)
   - Parameter rules (`AI_cup_parameter_info_cleaned_v2.csv`)
2. Check data quality and explain what is wrong.
3. Clean event timestamps for reliable machine+part timelines.
4. Plot understandable graphs (machine, part, scrap trend).
5. Train an LSTM model for future scrap-rate prediction.


In [ ]:
# Install TensorFlow only if missing (first run may take time)
# !pip -q install tensorflow matplotlib seaborn scikit-learn


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

pd.set_option('display.max_columns', 120)
sns.set_theme(style='whitegrid', context='talk')
np.random.seed(42)
tf.random.set_seed(42)


In [ ]:
ROOT = Path('..').resolve()
DATA_DIR = ROOT / 'backend_fastapi'
MES_PATH = DATA_DIR / 'MES_Manufacturing_M-231_M-356_M-471_M-607_M-612.xlsx - Data.csv'
PARAM_PATH = DATA_DIR / 'AI_cup_parameter_info_cleaned_v2.csv'

mes = pd.read_csv(MES_PATH)
params = pd.read_csv(PARAM_PATH)

print('MES shape:', mes.shape)
print('Params shape:', params.shape)
mes.head(2)


In [ ]:
def parse_mes_time_seconds(raw):
    if pd.isna(raw):
        return np.nan
    if hasattr(raw, 'hour') and hasattr(raw, 'minute') and hasattr(raw, 'second'):
        return int(raw.hour) * 3600 + int(raw.minute) * 60 + int(raw.second)
    if isinstance(raw, str):
        txt = raw.strip()
        if txt:
            parts = txt.split(':')
            if len(parts) == 3 and all(p.isdigit() for p in parts):
                hh, mm, ss = map(int, parts)
                if 0 <= hh < 48 and 0 <= mm < 60 and 0 <= ss < 60:
                    return hh * 3600 + mm * 60 + ss
            try:
                ts = pd.Timestamp(txt)
                return int(ts.hour) * 3600 + int(ts.minute) * 60 + int(ts.second)
            except Exception:
                pass
    try:
        num = int(float(raw))
    except Exception:
        return np.nan
    if 0 <= num <= 172800:
        return num
    txt = str(num).zfill(6)[-6:]
    hh, mm, ss = int(txt[:2]), int(txt[2:4]), int(txt[4:])
    if 0 <= hh < 24 and 0 <= mm < 60 and 0 <= ss < 60:
        return hh * 3600 + mm * 60 + ss
    return np.nan

def build_event_timestamp(df):
    out = df.copy()
    end_date = pd.to_datetime(out.get('machine_event_end_date'), errors='coerce')
    create_date = pd.to_datetime(out.get('machine_event_create_date'), errors='coerce')
    shift_ts = pd.to_datetime(out.get('plant_shift_timestamp'), errors='coerce')
    end_sec = out.get('machine_event_end_time', pd.Series(np.nan, index=out.index)).map(parse_mes_time_seconds)
    create_sec = out.get('machine_event_create_time', pd.Series(np.nan, index=out.index)).map(parse_mes_time_seconds)

    base_end = end_date.dt.floor('D')
    base_create = create_date.dt.floor('D')

    event_ts = base_end + pd.to_timedelta(end_sec.fillna(0), unit='s')
    no_end_sec = end_sec.isna()
    event_ts.loc[no_end_sec & end_date.notna()] = end_date.loc[no_end_sec & end_date.notna()]

    use_create = event_ts.isna()
    event_ts.loc[use_create] = (base_create + pd.to_timedelta(create_sec.fillna(0), unit='s')).loc[use_create]

    still_na = event_ts.isna()
    event_ts.loc[still_na] = shift_ts.loc[still_na]

    out['event_ts'] = pd.to_datetime(event_ts, errors='coerce')
    return out


In [ ]:
mes_clean = build_event_timestamp(mes)

quality = {}
quality['rows'] = len(mes_clean)
quality['machines'] = mes_clean['machine_id'].nunique()
quality['parts'] = mes_clean['part_number'].nunique(dropna=True)
quality['missing_part_ratio'] = float(mes_clean['part_number'].isna().mean())
quality['missing_event_ts_ratio'] = float(mes_clean['event_ts'].isna().mean())
quality['negative_yield_rows'] = int((pd.to_numeric(mes_clean['yield_quantity'], errors='coerce') < 0).sum())
quality['invalid_scrap_rows'] = int((pd.to_numeric(mes_clean['scrap_quantity'], errors='coerce') < 0).sum())

end_time_dt = pd.to_datetime(mes_clean['machine_event_end_time'], errors='coerce')
quality['end_time_year_1970_ratio'] = float((end_time_dt.dt.year == 1970).mean())

mes_parts = set(mes_clean['part_number'].dropna().astype(str).str.upper())
param_parts = set(params['part_number'].dropna().astype(str).str.upper())
specific_param_parts = {p for p in param_parts if p != 'GLOBAL'}
quality['part_specific_config_coverage'] = float(len(mes_parts & specific_param_parts) / max(1, len(mes_parts)))

quality_df = pd.DataFrame({'metric': list(quality.keys()), 'value': list(quality.values())})
quality_df


## Why some data is wrong

Typical issues this dataset shows:

- `machine_event_end_time` and `machine_event_create_time` are mostly `1970-...` placeholders (time-only encoded as fake date).
- `part_number` has missing rows.
- Some `yield_quantity` rows are negative (usually correction/reversal events).
- Parameter file has part-specific limits for only one part (`8-1419168-4`), while MES has many parts.

These directly affect part-scoped forecasting and seam continuity.


In [ ]:
work = mes_clean.copy()
work['machine_id'] = work['machine_id'].astype(str).str.strip()
work['part_number'] = work['part_number'].astype(str).str.strip().str.upper()

for col in ['yield_quantity', 'scrap_quantity', 'strokes_total_quantity', 'machine_status_seconds_quantity']:
    work[col] = pd.to_numeric(work[col], errors='coerce')

# Remove impossible negatives for target engineering
work['yield_quantity'] = work['yield_quantity'].clip(lower=0)
work['scrap_quantity'] = work['scrap_quantity'].clip(lower=0)

# Aggregate to hourly machine+part+hydra timeline
agg = (
    work.dropna(subset=['event_ts', 'machine_id'])
    .groupby([pd.Grouper(key='event_ts', freq='1h'), 'machine_id', 'part_number', 'hydra_bmk_number'], dropna=False)
    .agg({
        'yield_quantity': 'sum',
        'scrap_quantity': 'sum',
        'strokes_total_quantity': 'sum',
        'machine_status_seconds_quantity': 'sum',
    })
    .reset_index()
)

den = agg['yield_quantity'] + agg['scrap_quantity']
agg['scrap_rate'] = np.where(den > 0, agg['scrap_quantity'] / den, 0.0)
agg['scrap_rate'] = agg['scrap_rate'].clip(0, 1)
agg = agg.sort_values('event_ts')

print('Aggregated rows:', len(agg))
agg.head()


In [ ]:
# Pick the machine+part with most history for stable LSTM training
segment_counts = (
    agg.groupby(['machine_id', 'part_number'], dropna=False)['event_ts']
    .count()
    .sort_values(ascending=False)
)
best_machine, best_part = segment_counts.index[0]
print('Selected segment:', best_machine, best_part, 'rows=', int(segment_counts.iloc[0]))

segment = agg[(agg['machine_id'] == best_machine) & (agg['part_number'] == best_part)].copy()
segment = segment.sort_values('event_ts').reset_index(drop=True)

fig, ax = plt.subplots(1, 1, figsize=(16, 5))
ax.plot(segment['event_ts'], segment['scrap_rate'], color='#d62728', lw=1.8, label='scrap_rate')
ax.set_title(f'Scrap Rate Timeline | {best_machine} | {best_part}')
ax.set_xlabel('event_ts')
ax.set_ylabel('scrap_rate')
ax.legend()
plt.tight_layout()


In [ ]:
FEATURES = ['scrap_rate', 'yield_quantity', 'scrap_quantity', 'strokes_total_quantity', 'machine_status_seconds_quantity']
segment_features = segment[FEATURES].fillna(0.0).astype(float)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(segment_features)

def make_sequences(data_2d, seq_len=24, target_col=0):
    X, y = [], []
    for i in range(seq_len, len(data_2d)):
        X.append(data_2d[i-seq_len:i])
        y.append(data_2d[i, target_col])
    return np.asarray(X), np.asarray(y)

SEQ_LEN = 24
X, y = make_sequences(X_scaled, seq_len=SEQ_LEN, target_col=0)
print('Sequence tensors:', X.shape, y.shape)

n = len(X)
n_train = int(n * 0.70)
n_val = int(n * 0.15)

X_train, y_train = X[:n_train], y[:n_train]
X_val, y_val = X[n_train:n_train+n_val], y[n_train:n_train+n_val]
X_test, y_test = X[n_train+n_val:], y[n_train+n_val:]

print('train/val/test:', X_train.shape, X_val.shape, X_test.shape)


In [ ]:
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1),
])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

early_stop = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=80,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1,
)


In [ ]:
# Predict on test set (scaled scrap_rate), then inverse-scale only target column
y_pred_scaled = model.predict(X_test, verbose=0).reshape(-1)

target_mean = scaler.mean_[0]
target_std = scaler.scale_[0]

y_true = (y_test * target_std) + target_mean
y_pred = (y_pred_scaled * target_std) + target_mean

mae = mean_absolute_error(y_true, y_pred)
rmse = mean_squared_error(y_true, y_pred, squared=False)
print(f'Test MAE:  {mae:.6f}')
print(f'Test RMSE: {rmse:.6f}')

fig, ax = plt.subplots(1, 1, figsize=(16, 5))
ax.plot(y_true, label='Actual scrap_rate', color='#1f77b4')
ax.plot(y_pred, label='Predicted scrap_rate', color='#ff7f0e', alpha=0.9)
ax.set_title('LSTM Test Prediction: Actual vs Predicted')
ax.set_xlabel('Test step')
ax.set_ylabel('scrap_rate')
ax.legend()
plt.tight_layout()


In [ ]:
def recursive_forecast(model, last_window, steps, scaler):
    window = last_window.copy()
    preds_scaled = []
    for _ in range(steps):
        pred = model.predict(window[np.newaxis, :, :], verbose=0)[0, 0]
        preds_scaled.append(pred)
        next_row = window[-1].copy()
        next_row[0] = pred  # update scrap_rate only
        window = np.vstack([window[1:], next_row])

    target_mean = scaler.mean_[0]
    target_std = scaler.scale_[0]
    preds = np.array(preds_scaled) * target_std + target_mean
    return np.clip(preds, 0.0, 1.0)

FUTURE_STEPS = 24
last_window = X_scaled[-SEQ_LEN:]
future_pred = recursive_forecast(model, last_window, FUTURE_STEPS, scaler)

future_index = pd.date_range(segment['event_ts'].iloc[-1] + pd.Timedelta(hours=1), periods=FUTURE_STEPS, freq='1h')
future_df = pd.DataFrame({'event_ts': future_index, 'pred_scrap_rate': future_pred})
future_df.head()


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(16, 6))
tail = segment.tail(120)
ax.plot(tail['event_ts'], tail['scrap_rate'], label='Observed scrap_rate', color='#1f77b4', lw=2)
ax.plot(future_df['event_ts'], future_df['pred_scrap_rate'], label='Forecast scrap_rate', color='#d62728', lw=2)
ax.axvline(tail['event_ts'].iloc[-1], color='black', linestyle='--', label='Past -> Future seam')
ax.set_title(f'LSTM Future Forecast | {best_machine} | {best_part}')
ax.set_xlabel('event_ts')
ax.set_ylabel('scrap_rate')
ax.legend()
plt.tight_layout()


## Save model artifacts

Use these cells if you want to export trained artifacts for backend experiments.


In [ ]:
import joblib

OUT_DIR = ROOT / 'backend_fastapi' / 'models'
OUT_DIR.mkdir(parents=True, exist_ok=True)

model.save(OUT_DIR / 'lstm_scrap_rate_machine_part.h5')
joblib.dump(scaler, OUT_DIR / 'lstm_scrap_rate_scaler.joblib')

meta = {
    'machine_id': str(best_machine),
    'part_number': str(best_part),
    'features': FEATURES,
    'seq_len': SEQ_LEN,
    'future_steps': FUTURE_STEPS,
    'test_mae': float(mae),
    'test_rmse': float(rmse),
}
pd.Series(meta).to_json(OUT_DIR / 'lstm_scrap_rate_metadata.json', indent=2)
print('Saved artifacts to', OUT_DIR)
